In [ ]:
import subprocess, sys, os, glob, shutil, time, json
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(r.stdout[-2000:], r.stderr[-1500:], flush=True); return r
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
sh("free -g; nproc; df -h /kaggle/working | tail -1")
W = "/kaggle/working/w"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
for f in ("grpo_spice.py", "grpo_luoi.py", "build_branch_data.py", "grpo_tu_base.py", "prompt_keys_ck500.json"):
    k = [p for p in glob.glob(f"/kaggle/input/**/{f}", recursive=True) if "/notebooks/" not in p]
    assert k, f"DUNG: thieu {f}"; print(f, "<-", k[0], flush=True); shutil.copyfile(k[0], f"{W}/{f}")
sh(f"md5sum {W}/*.py")
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
ADAPTER = f"{BUNDLE}/adapter_s1_seed101"
print("BUNDLE", BUNDLE, os.listdir(ADAPTER), flush=True)
open(f"{W}/kiem_tu_base_cpu.py", "w", encoding="utf-8").write('# -*- coding: utf-8 -*-\n"""Kiểm lượt `tu=base` (grpo_tu_base.py) bằng mô hình THẬT trên Kaggle CPU, 7/11/2026 — 0 GPU.\n\nSelftest cũ chỉ nạp giả. Ở đây đi đúng đường GL.train dùng, dừng trước GRPOTrainer:\n  1. GS.nap qua bọc của grpo_tu_base: phải in `[xuất phát] Base …`, nạp đúng Qwen/Qwen2.5-VL-3B-Instruct\n     dù --merged trỏ s1_merged (fp32 trên CPU thay cho fp16 T4; mọi thứ khác như cũ).\n  2. LoRA mới cấu hình như grpo_luoi.train (r/alpha/target của S1): số tham số học = S1, không tham số thị giác,\n     lora_B toàn 0.\n  3. Với 2 câu nhắc thật (ảnh + OCR): logits khi bật adapter = khi tắt adapter ⇒ KL tới tham chiếu = 0 ở bước 0,\n     và tham chiếu đúng là Base (so với một lần nạp Base thứ hai không qua bọc, không LoRA).\n  4. GS.dung_hang(seed 101) ra đúng tập câu nhắc của ck500 (prompt_keys_ck500.json) nếu có tệp.\n  5. Greedy 2 câu từ Base (mốc độ dài câu, Base trung vị ~71 ký tự trên test).\n  python kiem_tu_base_cpu.py --bundle B --adapter A [--keys prompt_keys_ck500.json]\n"""\nimport argparse, json, os, sys, time, types\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport torch, transformers  # noqa: E402\nimport grpo_spice as GS     # noqa: E402\nimport grpo_tu_base as TB   # noqa: E402\n\nt0 = time.time()\nlog = lambda *a: print(f"[{time.time() - t0:6.0f} s]", *a, flush=True)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--bundle", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--keys", default="")\n    a = ap.parse_args()\n    assert not torch.cuda.is_available(), "phép kiểm này dành cho CPU"\n    GS._dtype = lambda: torch.float32\n    torch.cuda.get_device_name = lambda i=0: "CPU"\n    M = transformers.Qwen2_5_VLForConditionalGeneration\n    goc = M.from_pretrained.__func__\n\n    def fp(cls, path, *x, **kw):\n        kw["device_map"] = "cpu"\n        return goc(cls, path, *x, **kw)\n    M.from_pretrained = classmethod(fp)\n\n    TB.cai()\n    ns = types.SimpleNamespace(merged="/kaggle/working/s1_merged", q4=False)\n    proc, model, dt = GS.nap(ns)\n    assert ns.merged == GS.BASE and model.config._name_or_path == GS.BASE, (ns.merged, model.config._name_or_path)\n    log("✅ 1. nạp Base qua bọc tu=base", model.config._name_or_path, dt)\n\n    from peft import LoraConfig, get_peft_model\n    s1 = json.load(open(os.path.join(a.adapter, "adapter_config.json"), encoding="utf-8"))\n    lc = LoraConfig(r=s1["r"], lora_alpha=s1["lora_alpha"], lora_dropout=0.0, bias="none",\n                    target_modules=s1["target_modules"], task_type="CAUSAL_LM")\n    model = get_peft_model(model, lc)\n    ntr = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad)\n    vis = [n for n, p in model.named_parameters() if p.requires_grad and "visual" in n]\n    nb = [n for n, p in model.named_parameters() if "lora_B" in n and p.abs().max().item() != 0]\n    ns1 = GS._n_s1(a.adapter)\n    log(f"[LoRA mới] r {s1[\'r\']} · alpha {s1[\'lora_alpha\']} · học {ntr:,} · S1 {ns1:,} · thị giác học {len(vis)} · lora_B≠0 {len(nb)}")\n    assert ntr == ns1 and not vis and not nb\n    log("✅ 2. LoRA mới cùng cỡ S1, không học thị giác, B = 0")\n\n    rows, bo = GS.dung_hang(a.bundle, n=GS.N_PROMPT, seed=101)\n    keys = [r["key"] for r in rows]\n    log(f"[câu nhắc] {len(rows)} · bỏ {dict(bo)}")\n    if a.keys and os.path.exists(a.keys):\n        k0 = json.load(open(a.keys))\n        assert keys == k0, f"⛔ tập câu nhắc khác ck500 ({len(keys)} vs {len(k0)}, trùng {len(set(keys) & set(k0))})"\n        log("✅ 4. tập câu nhắc trùng thứ tự ck500")\n    else:\n        log("⚠️ 4. không có prompt_keys_ck500.json, bỏ qua")\n\n    from PIL import Image\n    model.eval()\n    proc.tokenizer.padding_side = "left"\n    inps = []\n    for r in rows[:2]:\n        msg = [{"role": "system", "content": GS.SYS},\n               {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": r["prompt"][1]["content"]}]}]\n        text = proc.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)\n        img = Image.open(r["image"]).convert("RGB")\n        inps.append((r, proc(text=[text], images=[img], return_tensors="pt")))\n    lg_on, lg_off = [], []\n    with torch.no_grad():\n        for r, inp in inps:\n            lg_on.append(model(**inp).logits[0, -8:].clone())\n            with model.disable_adapter():\n                lg_off.append(model(**inp).logits[0, -8:].clone())\n            log(f"  {r[\'key\']}: {inp[\'input_ids\'].shape[1]} token · max|Δlogit| bật−tắt adapter "\n                f"{(lg_on[-1] - lg_off[-1]).abs().max().item():.3e}")\n    assert all(torch.equal(x, y) for x, y in zip(lg_on, lg_off)), "⛔ adapter mới làm đổi logits ở bước 0"\n    log("✅ 3a. logits bật = tắt adapter (KL = 0 ở bước 0)")\n\n    with torch.no_grad():\n        for r, inp in inps:\n            L = inp["input_ids"].shape[1]\n            g = model.generate(**inp, max_new_tokens=48, do_sample=False, temperature=None, top_p=None, top_k=None)\n            s = proc.decode(g[0][L:], skip_special_tokens=True).strip()\n            log(f"  greedy Base {r[\'key\']}: {s!r} ({len(s)} ký tự) · câu chuẩn {r[\'gold\']!r}")\n    log("✅ 5. sinh được câu từ Base")\n    import gc\n    del model; gc.collect()   # hai bản fp32 3B không vừa 30 GB RAM\n    ref = M.from_pretrained(GS.BASE, device_map="cpu", **GS._kw(torch.float32))\n    ref.eval()\n    with torch.no_grad():\n        d = max((ref(**inp).logits[0, -8:] - x).abs().max().item() for (_, inp), x in zip(inps, lg_off))\n    log(f"  Base nạp thẳng (không bọc, không LoRA) − tham chiếu: max|Δlogit| {d:.3e}")\n    assert d < 1e-3, "⛔ tham chiếu KL không phải Base"\n    log("✅ 3b. tham chiếu KL = Base")\n    del ref\n\n    log("TONG: tu=base ĐẠT trên mô hình thật (CPU, fp32)")\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
t0 = time.time(); LOG = "/kaggle/working/kiem_tu_base_cpu.log"
env = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS="1", TQDM_DISABLE="1", TRANSFORMERS_VERBOSITY="warning")
p = subprocess.Popen(f"cd {W} && {sys.executable} -u kiem_tu_base_cpu.py --bundle {BUNDLE} --adapter {ADAPTER} --keys {W}/prompt_keys_ck500.json > {LOG} 2>&1", shell=True, env=env)
while p.poll() is None:
    time.sleep(60); print(time.strftime("%H:%M"), f"{(time.time()-t0)/60:.0f} ph", open(LOG, errors="ignore").read()[-300:].replace("\n", " | "), flush=True)
MA = p.returncode; print("===== ma thoat", MA, f"{time.time()-t0:.0f} s", flush=True); print(open(LOG, errors="ignore").read()[-8000:], flush=True)


In [ ]:
R = "/kaggle/working/ket_qua_tu_base"; os.makedirs(R, exist_ok=True); shutil.copyfile(LOG, f"{R}/kiem_tu_base_cpu.log")
shutil.make_archive("/kaggle/working/ket_qua_tu_base", "zip", R); shutil.rmtree(W)
assert MA == 0 and "TONG: tu=base" in open(LOG).read(), "KHONG DAT"
